In [9]:
from __future__ import annotations

import sys
from pathlib import Path


def _find_workspace_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root from the current working directory.")


WORKSPACE_DIR = _find_workspace_root(Path.cwd())
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
SHARED_ASSETS_DIR = paths["shared_assets"]
APPENDIX_DIR = CODE_DIR / "appendix"
APPENDIX_OUTPUTS_DIR = OUTPUTS_DIR / "appendix"

for candidate in [CODE_DIR, APPENDIX_DIR]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

print("workspace_root:", WORKSPACE_DIR)
print("appendix_outputs_root:", APPENDIX_OUTPUTS_DIR)


workspace_root: /Users/sallz/allzen/damadi/research/semantic-change/workspace
appendix_outputs_root: /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix


# Field-Specific, Bipartite, and Dark-Matter-Induced Discourse Graphs

This notebook writes three Gephi- and Cytoscape-ready graph families for the dark matter corpus:
- one field-specific discourse network for `astrophysics`
- one field-specific discourse network for `high-energy physics`
- one term-category bipartite graph linking salient terms to those two `arxiv_category` values
- one dark-matter-induced discourse network built from the literal `dark matter` subset, with full term-term edges inside that induced subcorpus

Design choices for this v1:
- the text corpus comes from the merged ADS + `dm_models.parquet` source already validated elsewhere
- node selection comes from the existing `keyness_unigrams.csv` and `keyness_bigrams.csv` tables
- `collostructional_full.csv` is intentionally not used as the main graph seed because its `A/B` labels are not `arxiv_category` values
- `primary_arxiv_class`-derived metadata is preserved as pipe-joined `arxiv_class` plus dominant-class summaries


In [10]:
import pandas as pd
from IPython.display import display

from appendix_discourse_graphs import (
    CATEGORY_CONFIG,
    build_bipartite_network,
    build_candidate_summary,
    build_dark_matter_induced_network,
    build_field_specific_network,
    build_stopwords,
    load_keyness_candidates,
    load_merged_papers,
)

ADS_PATH = DATA_DIR / "parquet" / "ADS_MAIN_FRAME_2025.parquet"
DM_MODELS_PATH = DATA_DIR / "parquet" / "dm_models.parquet"
KEYNESS_UNIGRAMS_PATH = RESEARCH_ROOT / "phenomena-particles-plurality" / "workspace" / "local" / "lexical-data" / "keyness_unigrams.csv"
KEYNESS_BIGRAMS_PATH = RESEARCH_ROOT / "phenomena-particles-plurality" / "workspace" / "local" / "lexical-data" / "keyness_bigrams.csv"
STOPWORDS_DIR = SHARED_ASSETS_DIR / "lists" / "tfidf"
SKLEARN_STOPWORDS_PATH = STOPWORDS_DIR / "sklearn_english_stopwords.txt"
FINAL_STOPWORDS_PATH = STOPWORDS_DIR / "final_stopwords.txt"
GRAPH_EXPORT_ROOT = APPENDIX_OUTPUTS_DIR / "graph_exports"
GRAPH_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

MAX_DOCS = None  # set to an integer like 5000 for a faster exploratory run
SELECTED_CATEGORIES = ("astrophysics", "high-energy physics")
MIN_TOTAL_DF = 25
TOP_UNIGRAMS_PER_CATEGORY = 200
TOP_BIGRAMS_PER_CATEGORY = 120
MIN_EDGE_DOC_SHARE = 0.0005
MIN_EDGE_COUNT_FLOOR = 3
MIN_EDGE_NPMI = 0.10

stopwords = build_stopwords(SKLEARN_STOPWORDS_PATH, FINAL_STOPWORDS_PATH)

print("ads source:", ADS_PATH)
print("dm_models source:", DM_MODELS_PATH)
print("keyness unigrams:", KEYNESS_UNIGRAMS_PATH)
print("keyness bigrams:", KEYNESS_BIGRAMS_PATH)
print("graph export root:", GRAPH_EXPORT_ROOT)
print("max docs:", MAX_DOCS if MAX_DOCS is not None else "full corpus")


ImportError: cannot import name 'build_dark_matter_induced_network' from 'appendix_discourse_graphs' (/Users/sallz/allzen/damadi/research/semantic-change/workspace/code/appendix/appendix_discourse_graphs.py)

In [3]:
papers = load_merged_papers(
    ads_path=ADS_PATH,
    dm_models_path=DM_MODELS_PATH,
    selected_categories=SELECTED_CATEGORIES,
    max_docs=MAX_DOCS,
)
candidates = load_keyness_candidates(
    keyness_unigrams_path=KEYNESS_UNIGRAMS_PATH,
    keyness_bigrams_path=KEYNESS_BIGRAMS_PATH,
    stopwords=stopwords,
    min_total_df=MIN_TOTAL_DF,
    top_unigrams_per_category=TOP_UNIGRAMS_PER_CATEGORY,
    top_bigrams_per_category=TOP_BIGRAMS_PER_CATEGORY,
)
candidate_summary = build_candidate_summary(candidates)

print("papers entering discourse-graph build:", len(papers))
print("papers by arxiv_category:")
display(papers["merged_arxiv_category"].value_counts().rename_axis("arxiv_category").reset_index(name="n_papers"))
print("selected candidate summary:")
display(candidate_summary)
print("sample candidates:")
display(
    candidates[
        [
            "arxiv_category",
            "term",
            "term_type",
            "source_table",
            "total_df",
            "salience",
            "field_doc_freq",
            "field_log_odds",
        ]
    ]
    .sort_values(["arxiv_category", "salience", "term"], ascending=[True, False, True])
    .groupby("arxiv_category")
    .head(15)
    .reset_index(drop=True)
)


papers entering discourse-graph build: 118356
papers by arxiv_category:


,arxiv_category,n_papers
0,astrophysics,78606
1,high-energy physics,39750


selected candidate summary:


,arxiv_category,term_type,terms,salience_min,salience_max
0,astrophysics,bigram,120,47.288529,71.031252
1,astrophysics,unigram,200,40.031755,94.878890
2,high-energy physics,bigram,120,39.361939,62.371491
3,high-energy physics,unigram,200,26.021189,59.711542


sample candidates:


,arxiv_category,term,term_type,source_table,total_df,salience,field_doc_freq,field_log_odds
0,astrophysics,star-forming,unigram,keyness_unigrams,3339,94.878890,3339,11.693627
1,astrophysics,early-type,unigram,keyness_unigrams,1867,81.767409,1867,10.855104
2,astrophysics,line-of-sight,unigram,keyness_unigrams,1190,72.281395,1190,10.205563
3,astrophysics,metallicities,unigram,keyness_unigrams,1173,71.988011,1173,10.184813
4,astrophysics,photometric,unigram,keyness_unigrams,3297,71.829770,3294,8.866700
5,astrophysics,photometric redshift,bigram,keyness_bigrams,1119,71.031252,1119,10.116850
6,astrophysics,outflows,unigram,keyness_unigrams,1114,70.940674,1114,10.110392
7,astrophysics,radial velocity,bigram,keyness_bigrams,1092,70.537944,1092,10.081629
8,astrophysics,massive cluster,bigram,keyness_bigrams,1055,69.844682,1055,10.031922
9,astrophysics,photometry,unigram,keyness_unigrams,1863,69.779485,1862,9.266273


## Field-Specific Discourse Networks

Each category-specific network uses a field-filtered subcorpus and a field-filtered salient-term vocabulary. Nodes are terms, and edges represent abstract-level co-occurrence within the given `arxiv_category`.


In [4]:
field_results = build_field_specific_network(
    papers=papers,
    candidates=candidates,
    export_root=GRAPH_EXPORT_ROOT,
    max_docs=MAX_DOCS,
    min_edge_doc_share=MIN_EDGE_DOC_SHARE,
    min_edge_count_floor=MIN_EDGE_COUNT_FLOOR,
    min_edge_npmi=MIN_EDGE_NPMI,
)

field_export_summary = pd.DataFrame(
    [
        {
            "arxiv_category": category,
            "export_dir": str(payload["export_dir"]),
            "retained_nodes": payload["manifest"]["retained_nodes"],
            "retained_edges": payload["manifest"]["retained_edges"],
            "n_documents": payload["manifest"]["n_documents"],
        }
        for category, payload in field_results.items()
    ]
).sort_values("arxiv_category").reset_index(drop=True)

display(field_export_summary)
for category, payload in field_results.items():
    print(f"\n{category} nodes preview")
    display(payload["nodes"].head(15))
    print(f"{category} edges preview")
    display(payload["edges"].head(15))


,arxiv_category,export_dir,retained_nodes,retained_edges,n_documents
0,astrophysics,/Users/sallz/allzen/damadi/research/semantic-c...,284,4182,78606
1,high-energy physics,/Users/sallz/allzen/damadi/research/semantic-c...,297,4969,39750



astrophysics nodes preview


,Id,Label,term_type,source_table,arxiv_category,arxiv_class,dominant_arxiv_class,dominant_arxiv_class_share,total_df,Astrophysics,...,field_probability,field_log_odds,salience,doc_freq_in_network,doc_share_in_network,degree,year_min,year_max,year_mean,year_median
0,stellar-mass,stellar-mass,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.660141,5474,5462,...,0.082784,7.759700,66.795095,7509,0.095527,134,1994,2026,2017.764416,2019.0
1,star-forming,star-forming,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.GA,0.670818,3339,3339,...,0.050610,11.693627,94.878890,4256,0.054143,121,1994,2026,2017.342105,2018.0
2,stellar,stellar,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.555176,10819,10611,...,0.160817,4.657648,43.265602,12877,0.163817,120,1972,2026,2015.604877,2017.0
3,metallicity,metallicity,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.583834,3274,3265,...,0.049489,7.413372,60.004373,3996,0.050836,116,1993,2026,2015.190190,2016.5
4,imaging,imaging,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.401126,3666,3634,...,0.055081,5.793384,47.547054,4440,0.056484,112,1993,2026,2015.859685,2017.0
5,photometric,photometric,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.369261,3297,3294,...,0.049928,8.866700,71.829770,4008,0.050988,104,1972,2026,2015.874002,2017.0
6,photometry,photometry,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.488164,1863,1862,...,0.028226,9.266273,69.779485,2239,0.028484,104,1993,2026,2015.016525,2016.0
7,spectroscopic,spectroscopic,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.CO,0.410792,4350,4324,...,0.065538,6.338608,53.105880,5504,0.070020,103,1993,2026,2017.656795,2019.0
8,populations,populations,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.498586,3494,3457,...,0.052398,5.514906,44.996606,4242,0.053965,95,1972,2026,2015.714286,2017.0
9,early-type,early-type,unigram,keyness_unigrams,astrophysics,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.468927,1867,1867,...,0.028302,10.855104,81.767409,2124,0.027021,89,1993,2026,2013.113936,2013.0


astrophysics edges preview


,Source,Target,count,npmi,source_doc_freq,target_doc_freq,year_min,year_max,year_mean,year_median,arxiv_class,dominant_arxiv_class,dominant_arxiv_class_share,arxiv_category,Weight
0,redshift,survey,7694,0.228806,20310,17497,1992,2026,2015.367039,2017.0,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.CO,0.502599,astrophysics,7694
1,spectroscopic,survey,3276,0.309509,5504,17497,1995,2026,2018.100122,2019.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.CO,0.471917,astrophysics,3276
2,stellar-mass,stellar,3104,0.286406,7509,12877,1995,2026,2017.434601,2018.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.683312,astrophysics,3104
3,stellar-mass,redshift,3021,0.135882,7509,20310,1996,2026,2017.729560,2019.0,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.GA,0.632241,astrophysics,3021
4,redshift,bias,2926,0.170861,20310,6454,1992,2026,2015.258715,2017.0,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.CO,0.628161,astrophysics,2926
5,survey,bias,2835,0.204599,17497,6454,1992,2026,2016.185538,2018.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.CO,0.643739,astrophysics,2835
6,redshift,spectroscopic,2809,0.204311,20310,5504,1994,2026,2017.444998,2019.0,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.CO,0.507654,astrophysics,2809
7,populations,stellar,2429,0.359926,4242,12877,1972,2026,2015.322767,2016.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.580486,astrophysics,2429
8,photometric,survey,2327,0.272370,4008,17497,1995,2026,2016.836700,2018.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.CO,0.418994,astrophysics,2327
9,imaging,survey,2166,0.218469,4440,17497,1995,2026,2016.719760,2018.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.448753,astrophysics,2166



high-energy physics nodes preview


,Id,Label,term_type,source_table,arxiv_category,arxiv_class,dominant_arxiv_class,dominant_arxiv_class_share,total_df,High-energy physics,...,field_probability,field_log_odds,salience,doc_freq_in_network,doc_share_in_network,degree,year_min,year_max,year_mean,year_median
0,yukawa,yukawa,unigram,keyness_unigrams,high-energy physics,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.925258,1639,1552,...,0.047442,5.160955,38.203722,1940,0.048805,119,1991,2026,2015.818041,2017.0
1,minimal,minimal,unigram,keyness_unigrams,high-energy physics,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.924859,4770,3987,...,0.121852,3.359268,28.454048,4964,0.124881,119,1992,2026,2014.366841,2015.0
2,charged,charged,unigram,keyness_unigrams,high-energy physics,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.868152,2972,2640,...,0.080690,4.001174,31.998695,3322,0.083572,114,1991,2026,2016.711017,2018.0
3,higgs boson,higgs boson,bigram,keyness_bigrams,high-energy physics,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.871269,2614,2593,...,0.079254,7.926209,62.371491,3216,0.080906,113,1992,2026,2015.822450,2016.0
4,boson,boson,unigram,keyness_unigrams,high-energy physics,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.862924,5070,4820,...,0.147308,5.278089,45.028928,5931,0.149208,109,1992,2026,2016.788231,2017.0
5,minimal supersymmetry,minimal supersymmetry,bigram,keyness_bigrams,high-energy physics,hep-ex|hep-ph|hep-th,hep-ph,0.949246,1768,1710,...,0.052271,5.881627,43.983802,2187,0.055019,108,1992,2026,2011.878372,2012.0
6,higgs,higgs,unigram,keyness_unigrams,high-energy physics,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.901121,7385,7223,...,0.220740,6.485975,57.772799,8829,0.222113,106,1991,2026,2015.872239,2016.0
7,doublet,doublet,unigram,keyness_unigrams,high-energy physics,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.959721,1799,1710,...,0.052271,5.268176,39.487831,2433,0.061208,106,1991,2026,2017.672421,2018.0
8,yukawa coupling,yukawa coupling,bigram,keyness_bigrams,high-energy physics,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.927259,1087,1060,...,0.032407,6.280958,43.917065,1306,0.032855,105,1993,2026,2015.455590,2016.0
9,singlet,singlet,unigram,keyness_unigrams,high-energy physics,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.971275,2232,2182,...,0.066694,6.445343,49.700696,2611,0.065686,104,1993,2026,2016.916890,2018.0


high-energy physics edges preview


,Source,Target,count,npmi,source_doc_freq,target_doc_freq,year_min,year_max,year_mean,year_median,arxiv_class,dominant_arxiv_class,dominant_arxiv_class_share,arxiv_category,Weight
0,higgs,boson,3875,0.463441,8829,5931,1992,2026,2015.994065,2016.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.879226,high-energy physics,3875
1,higgs,decay,3104,0.110661,8829,10539,1991,2026,2016.607925,2017.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.896263,high-energy physics,3104
2,higgs,coupling,3022,0.149248,8829,9262,1992,2026,2016.450364,2017.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.925546,high-energy physics,3022
3,breaking,symmetry,2903,0.495344,4660,6774,1992,2026,2015.706511,2017.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.890458,high-energy physics,2903
4,supersymmetry,minimal,2765,0.401346,7596,4964,1992,2026,2011.539602,2012.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.941049,high-energy physics,2765
5,gauge,symmetry,2722,0.303445,7080,6774,1992,2026,2016.576782,2018.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.864438,high-energy physics,2722
6,higgs,supersymmetry,2681,0.171760,8829,7596,1992,2026,2012.523685,2013.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.938083,high-energy physics,2681
7,gauge,coupling,2662,0.176988,7080,9262,1992,2026,2016.098798,2017.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.848986,high-energy physics,2662
8,boson,decay,2444,0.158112,5931,10539,1993,2026,2017.316285,2018.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.833879,high-energy physics,2444
9,higgs,symmetry,2346,0.156963,8829,6774,1993,2026,2016.149190,2017.0,hep-ex|hep-lat|hep-ph|hep-th,hep-ph,0.936061,high-energy physics,2346


## Term-Category Bipartite Graph

This export links the retained salient terms to `astrophysics` or `high-energy physics` using the existing keyness association scores. It complements the field-specific discourse maps by showing lexical field affiliation directly.


In [5]:
bipartite_result = build_bipartite_network(
    papers=papers,
    candidates=candidates,
    export_root=GRAPH_EXPORT_ROOT,
    max_docs=MAX_DOCS,
)

bipartite_export_summary = pd.DataFrame(
    [
        {
            "export_dir": str(bipartite_result["export_dir"]),
            "retained_term_nodes": bipartite_result["manifest"]["retained_term_nodes"],
            "category_nodes": bipartite_result["manifest"]["category_nodes"],
            "retained_edges": bipartite_result["manifest"]["retained_edges"],
        }
    ]
)

display(bipartite_export_summary)
print("bipartite nodes preview")
display(bipartite_result["nodes"].head(20))
print("bipartite edges preview")
display(bipartite_result["edges"].head(20))


/Users/sallz/allzen/damadi/research/semantic-change/workspace/code/appendix/appendix_discourse_graphs.py:853: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  nodes = pd.concat([pd.DataFrame(term_nodes), pd.DataFrame(category_nodes)], ignore_index=True)


,export_dir,retained_term_nodes,category_nodes,retained_edges
0,/Users/sallz/allzen/damadi/research/semantic-c...,602,2,602


bipartite nodes preview


,Id,Label,node_type,term_type,source_table,total_df,Astrophysics,High-energy physics,p_astro,p_hep,...,salience,dominant_category,dominant_log_odds,year_min,year_max,year_mean,year_median,arxiv_class,dominant_arxiv_class,dominant_arxiv_class_share
0,star-forming,star-forming,term,unigram,keyness_unigrams,3339,3339,0,0.050610,0.000015,...,94.878890,astrophysics,11.693627,1994,2026,2017.340615,2018.0,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.GA,0.670660
1,early-type,early-type,term,unigram,keyness_unigrams,1867,1867,0,0.028302,0.000015,...,81.767409,astrophysics,10.855104,1993,2026,2013.113936,2013.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.468927
2,metallicities,metallicities,term,unigram,keyness_unigrams,1173,1173,0,0.017784,0.000015,...,71.988011,astrophysics,10.184813,1994,2026,2015.583333,2017.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.604861
3,photometric,photometric,term,unigram,keyness_unigrams,3297,3294,3,0.049928,0.000107,...,71.829770,astrophysics,8.866700,1972,2026,2015.878863,2017.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.368893
4,photometric redshift,photometric redshift,term,bigram,keyness_bigrams,1119,1119,0,0.016966,0.000015,...,71.031252,astrophysics,10.116850,1998,2026,2016.083770,2017.0,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.CO,0.506358
5,outflows,outflows,term,unigram,keyness_unigrams,1114,1114,0,0.016890,0.000015,...,70.940674,astrophysics,10.110392,1994,2026,2017.100282,2018.0,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.GA,0.620056
6,radial velocity,radial velocity,term,bigram,keyness_bigrams,1092,1092,0,0.016557,0.000015,...,70.537944,astrophysics,10.081629,1993,2026,2014.507416,2016.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.509758
7,massive cluster,massive cluster,term,bigram,keyness_bigrams,1055,1055,0,0.015996,0.000015,...,69.844682,astrophysics,10.031922,1994,2026,2014.054237,2015.0,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.CO,0.421469
8,photometry,photometry,term,unigram,keyness_unigrams,1863,1862,1,0.028226,0.000046,...,69.779485,astrophysics,9.266273,1993,2026,2015.012054,2016.0,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.487946
9,major merger,major merger,term,bigram,keyness_bigrams,960,960,0,0.014556,0.000015,...,67.964464,astrophysics,9.895853,1996,2026,2014.782810,2015.0,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|h...,astro-ph.GA,0.488909


bipartite edges preview


,Source,Target,Weight,salience,term_type,source_table,Astrophysics,High-energy physics,p_astro,p_hep,log_odds_astro,log_odds_hep
0,star-forming,astrophysics,94.878890,94.878890,unigram,keyness_unigrams,3339,0,0.050610,0.000015,11.693627,-11.693627
1,early-type,astrophysics,81.767409,81.767409,unigram,keyness_unigrams,1867,0,0.028302,0.000015,10.855104,-10.855104
2,metallicities,astrophysics,71.988011,71.988011,unigram,keyness_unigrams,1173,0,0.017784,0.000015,10.184813,-10.184813
3,photometric,astrophysics,71.829770,71.829770,unigram,keyness_unigrams,3294,3,0.049928,0.000107,8.866700,-8.866700
4,photometric redshift,astrophysics,71.031252,71.031252,bigram,keyness_bigrams,1119,0,0.016966,0.000015,10.116850,-10.116850
5,outflows,astrophysics,70.940674,70.940674,unigram,keyness_unigrams,1114,0,0.016890,0.000015,10.110392,-10.110392
6,radial velocity,astrophysics,70.537944,70.537944,bigram,keyness_bigrams,1092,0,0.016557,0.000015,10.081629,-10.081629
7,massive cluster,astrophysics,69.844682,69.844682,bigram,keyness_bigrams,1055,0,0.015996,0.000015,10.031922,-10.031922
8,photometry,astrophysics,69.779485,69.779485,unigram,keyness_unigrams,1862,1,0.028226,0.000046,9.266273,-9.266273
9,major merger,astrophysics,67.964464,67.964464,bigram,keyness_bigrams,960,0,0.014556,0.000015,9.895853,-9.895853


## Dark-Matter-Induced Discourse Network

This export filters the corpus to papers containing the literal string `dark matter`, keeps the salient vocabulary, and then builds the full co-occurrence network among those terms inside that induced subcorpus. Node size can therefore be mapped to term frequency in the dark-matter subset, while node colour can use the HEP-vs-Astro `log_share_ratio`.


In [12]:
dark_matter_result = build_dark_matter_induced_network(
    papers=papers,
    candidates=candidates,
    export_root=GRAPH_EXPORT_ROOT,
    max_docs=MAX_DOCS,
    min_edge_doc_share=MIN_EDGE_DOC_SHARE,
    min_edge_count_floor=MIN_EDGE_COUNT_FLOOR,
    min_edge_npmi=MIN_EDGE_NPMI,
)

dark_matter_export_summary = pd.DataFrame(
    [
        {
            "export_dir": str(dark_matter_result["export_dir"]),
            "n_documents": dark_matter_result["manifest"]["n_documents"],
            "n_documents_astro": dark_matter_result["manifest"]["n_documents_astro"],
            "n_documents_hep": dark_matter_result["manifest"]["n_documents_hep"],
            "retained_nodes": dark_matter_result["manifest"]["retained_nodes"],
            "retained_edges": dark_matter_result["manifest"]["retained_edges"],
        }
    ]
)

display(dark_matter_export_summary)
print("dark matter subset papers:", len(dark_matter_result["subset_papers"]))
print("dark-matter-induced nodes preview")
display(dark_matter_result["nodes"].head(20))
print("dark-matter-induced edges preview")
display(dark_matter_result["edges"].head(20))


,export_dir,n_documents,n_documents_astro,n_documents_hep,retained_nodes,retained_edges
0,/Users/sallz/allzen/damadi/research/semantic-c...,43938,28062,15876,598,7959


dark matter subset papers: 43938
dark-matter-induced nodes preview


,Id,Label,term_type,source_table,total_df,Astrophysics,High-energy physics,p_astro,p_hep,log_odds_astro,...,log_share_ratio,degree,arxiv_class,dominant_arxiv_class,dominant_arxiv_class_share,dominant_category,year_min,year_max,year_mean,year_median
0,dark matter,dark matter,anchor,literal_anchor,36423,21375,15048,NaN,NaN,NaN,...,0.315392,597,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,hep-ph,0.365950,all,1992,2026,2015.557999,2017.0
1,redshift,redshift,unigram,keyness_unigrams,16843,16670,173,0.252641,0.005302,5.574429,...,-5.279699,162,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.CO,0.541157,astrophysics,1992,2026,2014.359594,2015.0
2,cluster,cluster,unigram,keyness_unigrams,13648,13394,254,0.202993,0.007777,4.706042,...,-3.714073,146,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.CO,0.375437,astrophysics,1993,2026,2012.639845,2013.0
3,decay,decay,unigram,keyness_unigrams,10191,1798,8393,0.027256,0.256494,-3.234267,...,2.532272,203,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,hep-ph,0.707516,high-energy physics,1992,2026,2016.238060,2017.0
4,survey,survey,unigram,keyness_unigrams,14614,14310,304,0.216875,0.009305,4.542699,...,-4.355266,141,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.CO,0.530886,astrophysics,1992,2026,2015.769231,2017.0
5,coupling,coupling,unigram,keyness_unigrams,9557,2054,7503,0.031136,0.229297,-2.880565,...,2.373671,148,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,hep-ph,0.674672,high-energy physics,1992,2026,2017.559746,2019.0
6,stellar,stellar,unigram,keyness_unigrams,10819,10611,208,0.160817,0.006371,4.657648,...,-3.963320,172,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.485563,astrophysics,1993,2026,2015.169634,2016.0
7,relic,relic,unigram,keyness_unigrams,4203,765,3438,0.011601,0.105076,-3.179094,...,3.745635,191,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,hep-ph,0.867426,high-energy physics,1993,2026,2016.410325,2017.0
8,higgs,higgs,unigram,keyness_unigrams,7385,162,7223,0.002463,0.220740,-6.485975,...,5.813680,198,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|h...,hep-ph,0.919566,high-energy physics,1993,2026,2015.975230,2016.0
9,symmetry,symmetry,unigram,keyness_unigrams,6403,871,5532,0.013208,0.169066,-3.678148,...,3.532518,148,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,hep-ph,0.829050,high-energy physics,1993,2026,2016.734608,2018.0


dark-matter-induced edges preview


,Source,Target,count_total,count_astro,count_hep,share_astro,share_hep,log_share_ratio,npmi_total,source_doc_freq,target_doc_freq,arxiv_class,dominant_arxiv_class,dominant_arxiv_class_share,year_min,year_max,year_mean,year_median,Weight
0,dark matter,redshift,6791,6694,97,0.238543,0.006110,-5.279699,0.00000,43938,6791,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,astro-ph.CO,0.541157,1992,2026,2014.359594,2015.0,6791
1,dark matter,cluster,5431,5207,224,0.185553,0.014109,-3.714073,0.00000,43938,5431,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.CO,0.375437,1993,2026,2012.639845,2013.0,5431
2,dark matter,decay,5402,1264,4138,0.045043,0.260645,2.532272,0.00000,43938,5402,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,hep-ph,0.707516,1992,2026,2016.238060,2017.0,5402
3,dark matter,survey,5148,5010,138,0.178533,0.008692,-4.355266,0.00000,43938,5148,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.CO,0.530886,1992,2026,2015.769231,2017.0,5148
4,dark matter,coupling,5038,1281,3757,0.045649,0.236647,2.373671,0.00000,43938,5038,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,hep-ph,0.674672,1992,2026,2017.559746,2019.0,5038
5,dark matter,stellar,4156,4011,145,0.142934,0.009133,-3.963320,0.00000,43938,4156,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,astro-ph.GA,0.485563,1993,2026,2015.169634,2016.0,4156
6,dark matter,relic,4126,480,3646,0.017105,0.229655,3.745635,0.00000,43938,4126,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,hep-ph,0.867426,1993,2026,2016.410325,2017.0,4126
7,dark matter,higgs,3593,109,3484,0.003884,0.219451,5.813680,0.00000,43938,3593,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|h...,hep-ph,0.919566,1993,2026,2015.975230,2016.0,3593
8,dark matter,symmetry,3346,443,2903,0.015786,0.182855,3.532518,0.00000,43938,3346,astro-ph|astro-ph.CO|astro-ph.EP|astro-ph.GA|a...,hep-ph,0.829050,1993,2026,2016.734608,2018.0,3346
9,dark matter,sector,3312,615,2697,0.021916,0.169879,2.953523,0.00000,43938,3312,astro-ph|astro-ph.CO|astro-ph.GA|astro-ph.HE|a...,hep-ph,0.760568,1992,2026,2017.680254,2019.0,3312
